# XGBoost Hyperparameter Tuning

This notebook demonstrates the hyperparameter tuning process for the XGBoost model used in the Culture Intelligence System. The goal is to find the optimal set of hyperparameters to improve model performance.

In [ ]:
import pandas as pd
import numpy as np
import xgboost as xgb
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.metrics import mean_absolute_error
from pathlib import Path

## 1. Load Processed Data
Loading the `culture_intelligence_v1.parquet` dataset, which contains the engineered features.

In [ ]:
data_path = Path('../src/data/processed/culture_intelligence_v1.parquet')
df = pd.read_parquet(data_path)
print(f"Processed dataset loaded successfully with {df.shape[0]} rows and {df.shape[1]} columns.")

## 2. Feature Selection and Data Split
Selecting the features for training and splitting the data into training and validation sets.

In [ ]:
features = ['culture_values', 'belonging_score', 'career_opp']
target = 'overall_rating'

X = df[features]
y = df[target]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

## 3. Hyperparameter Tuning with GridSearchCV
Defining a parameter grid and using GridSearchCV to find the best hyperparameters for the XGBoost Regressor.

In [ ]:
param_grid = {
    'n_estimators': [100, 200],
    'max_depth': [3, 5],
    'learning_rate': [0.1, 0.05],
    'subsample': [0.8, 1.0],
    'colsample_bytree': [0.8, 1.0]
}

xgb_reg = xgb.XGBRegressor(objective='reg:squarederror', random_state=42)

grid_search = GridSearchCV(estimator=xgb_reg, param_grid=param_grid, cv=3, scoring='neg_mean_absolute_error', verbose=2, n_jobs=-1)

grid_search.fit(X_train, y_train)

## 4. Best Parameters and Model Evaluation
Displaying the best parameters found by GridSearchCV and evaluating the model with these parameters.

In [ ]:
print("Best parameters found: ", grid_search.best_params_)

best_model = grid_search.best_estimator_
y_pred = best_model.predict(X_test)
mae = mean_absolute_error(y_test, y_pred)

print(f"Mean Absolute Error on the test set: {mae:.4f}")